# License Plate Detection — Faster R-CNN (Google Colab)
**CMPS 261 — Machine Learning Project**

Run on Colab **T4 GPU**. Make sure `license_plate_data.zip` is in your Google Drive root.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Faster R-CNN — License Plate Detection  (CMPS 261)
# ─────────────────────────────────────────────────────────────────────────────

# 1. INSTALL
import subprocess
subprocess.run(['pip', 'install', 'pycocotools', '-q'], check=True)

# 2. MOUNT DRIVE & EXTRACT DATA
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
zip_path = '/content/drive/MyDrive/license_plate_data.zip'
if not os.path.exists('/content/data/yolo'):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('Extracted.')
else:
    print('Already extracted.')

TRAIN_IMG = '/content/data/yolo/images/train'
VAL_IMG   = '/content/data/yolo/images/val'
TEST_IMG  = '/content/data/yolo/images/test'
TRAIN_LBL = '/content/data/yolo/labels/train'
VAL_LBL   = '/content/data/yolo/labels/val'
TEST_LBL  = '/content/data/yolo/labels/test'
print(f'Train: {len(os.listdir(TRAIN_IMG))} images | Val: {len(os.listdir(VAL_IMG))} | Test: {len(os.listdir(TEST_IMG))}')

# 3. GPU CHECK
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}' + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))

# 4. DATASET CLASS  (reads YOLO txt labels → Faster R-CNN xyxy format)
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as F

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'):
                continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx - w / 2) * W)
                ymin = max(0.0, (cy - h / 2) * H)
                xmax = min(float(W), (cx + w / 2) * W)
                ymax = min(float(H), (cy + h / 2) * H)
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
        if not boxes:
            boxes = [[0.0, 0.0, 1.0, 1.0]]
        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        target = {
            'boxes'   : boxes,
            'labels'  : labels,
            'image_id': torch.tensor([idx]),
            'area'    : (boxes[:,3]-boxes[:,1]) * (boxes[:,2]-boxes[:,0]),
            'iscrowd' : torch.zeros(len(boxes), dtype=torch.int64),
        }
        return F.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = LicensePlateDataset(TRAIN_IMG, TRAIN_LBL)
val_ds   = LicensePlateDataset(VAL_IMG,   VAL_LBL)
test_ds  = LicensePlateDataset(TEST_IMG,  TEST_LBL)
print(f'Loaded — Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

# 5. BUILD MODEL
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
model.to(DEVICE)

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
print('Model ready.')

# 6. TRAIN
import time

MODEL_PATH   = '/content/fasterrcnn_best.pth'
best_val     = float('inf')
train_losses, val_losses = [], []
NUM_EPOCHS   = 30

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    model.train()
    total_train = 0
    for images, targets in train_loader:
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item()
    train_loss = total_train / len(train_loader)

    model.train()  # keep train mode to get losses on val
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            total_val += sum(loss_dict.values()).item()
    val_loss = total_val / len(val_loader)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    flag = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        flag = ' <- best'

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.0f}s{flag}')

print('\nTraining complete!')

# 7. EVALUATE
import numpy as np

def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-6)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

tp, fp, fn = 0, 0, 0
iou_scores = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(DEVICE) for img in images]
        preds  = model(images)
        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= 0.5].cpu().numpy()
            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched:
                    tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched)

precision = tp / (tp + fp + 1e-6)
recall    = tp / (tp + fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)
mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0

print(f'\n=== Test Results ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

# 8. SAVE METRICS & DOWNLOAD
import json
from google.colab import files

metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN v2)',
    'epochs'   : NUM_EPOCHS,
    'precision': round(precision, 4),
    'recall'   : round(recall, 4),
    'f1'       : round(f1, 4),
    'mean_iou' : round(mean_iou, 4),
}
with open('/content/fasterrcnn_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

files.download('/content/fasterrcnn_best.pth')
files.download('/content/fasterrcnn_metrics.json')